# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their IDs. Here, we show all record sets, and for each record set, the associated fields (columns).

In [ ]:
# Explore record sets and their fields
all_record_sets = list(dataset.record_sets)

if not all_record_sets:
    print("No record sets found in the Croissant metadata. Please check your dataset or Croissant schema.")
else:
    for rec_set in all_record_sets:
        print(f"Record Set Name: {rec_set.name}")
        print(f"  @id: {rec_set.id}")
        print("  Fields (columns):")
        for field in rec_set.fields:
            print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")
        print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# Extract data for each record set (referenced by @id)
dataframes = {}
available_record_sets = list(dataset.record_sets)
record_set_ids = [rs.id for rs in available_record_sets]
print(f"Record sets found: {record_set_ids}")

# Load data for each available record set
for rs in available_record_sets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"Loaded {len(df)} records for Record Set @id '{rs.id}' ({rs.name})")

# For demonstration, select the main record set ('cr:Kilifi-MentalHealthSurvey') if available
main_record_set_id = None
for rs in available_record_sets:
    if 'kilifi' in rs.id.lower():
        main_record_set_id = rs.id
        break
if not main_record_set_id and record_set_ids:
    main_record_set_id = record_set_ids[0]  # fallback to first

if main_record_set_id:
    print(f"\nColumns for main record set '@id': {main_record_set_id}")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No main record set found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section gives examples using the main record set and selected fields.

In [ ]:
# Example: Filter and normalize the GAD-7 field if present
# Identify some common numeric fields by @id from the survey (change as per your schema record set field IDs)
numeric_fields = ['cr:GAD7_total_score', 'cr:PHQ9_total_score', 'cr:PSQ_total_score']

if main_record_set_id:
    df = dataframes[main_record_set_id].copy()
    found_numeric = None
    for field in numeric_fields:
        if field in df.columns:
            found_numeric = field
            break
    if found_numeric:
        threshold = 10
        filtered_df = df[df[found_numeric] > threshold]
        print(f"Filtered records with {found_numeric} > {threshold} (n={filtered_df.shape[0]}):")
        display(filtered_df.head())

        normalized_col = f"{found_numeric}_normalized"
        filtered_df[normalized_col] = (filtered_df[found_numeric] - filtered_df[found_numeric].mean()) / filtered_df[found_numeric].std()
        print(f"\nNormalized values for {found_numeric}:")
        display(filtered_df[[found_numeric, normalized_col]].head())

        group_field = 'cr:age_group'
        if group_field in df.columns:
            grouped_df = filtered_df.groupby(group_field)[found_numeric].mean().reset_index()
            print(f"\nMean {found_numeric} grouped by {group_field}:")
            display(grouped_df.head())
        else:
            print(f"Grouping field '{group_field}' not found in columns: {df.columns.tolist()}")
    else:
        print(f"None of the expected numeric fields {numeric_fields} found in columns: {df.columns.tolist()}")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here we plot the distribution of a numeric mental health score if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and found_numeric:
    plt.figure(figsize=(8, 5))
    sns.histplot(dataframes[main_record_set_id][found_numeric], kde=True, bins=20)
    plt.title(f"Distribution of {found_numeric}")
    plt.xlabel(found_numeric)
    plt.ylabel("Frequency")
    plt.show()

    # Optional: Boxplot by group if group column exists
    group_field = 'cr:age_group'
    if group_field in dataframes[main_record_set_id]:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=group_field, y=found_numeric, data=dataframes[main_record_set_id])
        plt.title(f"{found_numeric} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(found_numeric)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to load a Croissant metadata-defined dataset with `mlcroissant`, enumerate its record sets and fields (referencing all by `@id`), load records into pandas DataFrames, filter and normalize data, and create basic visualizations. For a deeper or more customized analysis, consult the field and record set `@id`s discovered in Section 2 and the [mlcroissant documentation](https://mlcommons.github.io/croissant/) for advanced extraction and transformation techniques.